In [ ]:
import cv2
from pathlib import Path
import shutil

def resample_vid(input_path, output_path, target_fps):
    cap = cv2.VideoCapture(str(input_path))
    
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {input_path}")
    
    # Get original properties
    original_fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Setup output video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(str(output_path), fourcc, target_fps, (width, height))
    
    # Calculate frame sampling rate
    frame_interval = original_fps / target_fps
    
    frame_idx = 0
    next_frame_to_write = 0.0
    frames_written = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Write this frame if the next interval is reached
        if frame_idx >= next_frame_to_write:
            out.write(frame)
            frames_written += 1
            next_frame_to_write += frame_interval
        
        frame_idx += 1
    
    cap.release()
    out.release()
    
    return True, original_fps, frames_written  # True = was resampled

# MAIN
input_dir = "vids"
output_dir = "vids_30fps"
input_path = Path(input_dir)
output_path = Path(output_dir)

# FPS Configuration
target_fps = 30 # Target frame rate
min_fps = 24 # Skip videos below this threshold

# Find all videos recursively
vid_extensions = ['.mp4', '.avi', '.mov', '.MP4', '.AVI', '.MOV']
vid_files = []
for ext in vid_extensions:
    vid_files.extend(input_path.rglob(f'*{ext}'))

print(f"Found {len(vid_files)} videos to process")
print(f"Target FPS: {target_fps}")
print(f"Minimum FPS threshold: {min_fps} (videos below this will be skipped)")

# Ask for confirmation
response = input("Continue? (y/n): ")
if response.lower() != 'y':
    print("Aborted.")
    exit()

print("\nStarting processing...\n")

stats = {
    'resampled': 0,
    'copied': 0,
    'skipped_corrupt': 0,
    'skipped_low_fps': 0,
    'errors': 0
}

for i, vid_path in enumerate(vid_files, 1):
    # Find relative path to preserve folder structure
    rel_path = vid_path.relative_to(input_path)
    output_file = output_path / rel_path
    
    # Create output directory if needed
    output_file.parent.mkdir(parents=True, exist_ok=True)
    
    try:
        # Check for corrupted fps
        cap = cv2.VideoCapture(str(vid_path))
        original_fps = cap.get(cv2.CAP_PROP_FPS)
        cap.release()
        
        # Skip videos with suspicious fps (corrupted)
        if original_fps > 1000 or original_fps < 1:
            print(f"  ! SKIPPED: Corrupted fps ({original_fps})\n")
            stats['skipped_corrupt'] += 1
            continue
        
        # Skip videos below minimum fps threshold
        if original_fps < min_fps:
            print(f"  ! SKIPPED: FPS too low ({original_fps:.1f} < {min_fps})\n")
            stats['skipped_low_fps'] += 1
            continue
        
        # Check if already at target fps
        if abs(original_fps - target_fps) < 0.1:
            # if yes, copy
            shutil.copy2(vid_path, output_file)
            print(f" Copied! (already {target_fps} fps)\n")
            stats['copied'] += 1
        else:
            # Resample
            was_resampled, orig_fps, new_frames = resample_vid(
                vid_path, 
                output_file, 
                target_fps
            )
            print(f"  Resampled! {orig_fps:.1f} fps → {target_fps} fps ({new_frames} frames)\n")
            stats['resampled'] += 1
            
    except Exception as e:
        print(f"  ERROR: {e}\n")
        stats['errors'] += 1
        continue
    
    # Print progress summary every 100 videos
    if i % 100 == 0:
        processed = stats['resampled'] + stats['copied']
        print(f"\n--- Progress: {i}/{len(vid_files)} ({i/len(vid_files)*100:.1f}%) ---")
        print(f"    Successfully processed: {processed}")
        print(f"    Skipped: {stats['skipped']}, Errors: {stats['errors']}\n")

# Print summary
print("\n" + "=" * 70)
print("PROCESSING COMPLETE")
print("=" * 70)
print(f"Total videos found: {len(vid_files)}")
print(f"  Resampled: {stats['resampled']} videos")
print(f"  Copied (already 30fps): {stats['copied']} videos")
print(f"  Skipped (corrupted fps): {stats['skipped_corrupt']} videos")
print(f"  Skipped (fps < {min_fps}): {stats['skipped_low_fps']} videos")
print(f"  Errors: {stats['errors']} videos")
print(f"\nSuccessfully processed: {stats['resampled'] + stats['copied']}/{len(vid_files)} videos")
print(f"Output directory: {output_path}")